# 模型阶段 CPU 准备：不下载权重，不启动 GPU

Actor admission 已经在 f32fd53 的双 T4 上通过。这次 Run All 做 plan、clone-comfy、install-model-deps、audit。
不把那次通过当成模型加载或视频成功。不重复 actor 探针，不下载四份权重。
ComfyUI 固定 2fcb6cef9ab1b6dc556b7cff1765a0724ad5e884，写到 /tmp/ComfyUI-h3，不写进只读输入包。
f481 的 Kaggle CPU 审计缺 comfy_aimdo，所以 comfy.model_management 导入失败。install-model-deps 会安装它，并且拒绝改动 torch、torchvision、torchaudio、ray。
结束后看三个 JSON 和 FileLink。缺文件会打印 MISSING。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ENTRY_SCRIPT = Path("scripts") / "run_actor_session.py"
MARKER_SCRIPT = Path("scripts") / "bootstrap_actor_admission.py"
MAX_DEPTH = 6


def default_seeds() -> list[Path]:
    return [Path.cwd(), Path("/kaggle/input"), Path("/kaggle/working")]


def find_package_root(seeds: list[Path] | None = None, environ: dict[str, str] | None = None) -> Path:
    """Return the only directory that contains the current admission entry.

    ``H3_T4_PACKAGE_ROOT`` selects one directory and disables the search.
    A modern Kaggle mount is ``/kaggle/input/datasets/<owner>/<dataset>/<package>``.
    Older sessions mounted the package directly under ``/kaggle/input`` or
    ``/kaggle/working``. More than one match is an error.
    """

    env = os.environ if environ is None else environ
    pinned = env.get("H3_T4_PACKAGE_ROOT", "").strip()
    if pinned:
        root = Path(pinned)
        if not _is_admission_package(root):
            raise SystemExit(f"H3_T4_PACKAGE_ROOT is not the admission package: {root}")
        return root.resolve()
    found: list[Path] = []
    seen: set[str] = set()
    for seed in seeds if seeds is not None else default_seeds():
        if not seed.exists():
            continue
        for match in _scan(seed.resolve(), MAX_DEPTH):
            key = str(match.resolve())
            if key not in seen:
                seen.add(key)
                found.append(match.resolve())
    if not found:
        raise SystemExit(
            "admission package not found. Upload the folder that contains "
            "scripts/run_actor_session.py or set H3_T4_PACKAGE_ROOT."
        )
    if len(found) > 1:
        lines = "\n".join(str(path) for path in found)
        raise SystemExit("multiple admission packages found; set H3_T4_PACKAGE_ROOT to one:\n" + lines)
    return found[0]


def _is_admission_package(root: Path) -> bool:
    return (root / ENTRY_SCRIPT).is_file() and (root / MARKER_SCRIPT).is_file()


def _scan(seed: Path, max_depth: int) -> list[Path]:
    matches: list[Path] = []
    for current, dirnames, _filenames in os.walk(seed):
        current_path = Path(current)
        try:
            depth = len(current_path.relative_to(seed).parts)
        except ValueError:
            depth = max_depth + 1
        if depth > max_depth:
            dirnames.clear()
            continue
        if _is_admission_package(current_path):
            matches.append(current_path)
            dirnames.clear()
    return matches

root = find_package_root()
evidence = Path("/kaggle/working/h3-model-prepare")
evidence.mkdir(parents=True, exist_ok=True)
script = root / "scripts" / "prepare_model_stage.py"
reports = []
for stage in ("plan", "clone-comfy", "install-model-deps", "audit"):
    report = evidence / (stage + ".json")
    code = subprocess.call([sys.executable, str(script), "--stage", stage, "--report", str(report), "--comfy-dir", "/tmp/ComfyUI-h3"], cwd=str(root))
    text = report.read_text(encoding="utf-8") if report.is_file() else "MISSING " + report.name
    print("STAGE", stage, "exit", code)
    print(text)
    reports.append(report)
print("download", False)
print("video_inference_claimed", False)
print("actor_admission_passed", False)
try:
    from IPython.display import FileLink, display
except Exception as exc:
    display = None
    print("filelink_unavailable", type(exc).__name__)
if display is not None:
    for report in reports:
        if report.is_file():
            display(FileLink(str(report)))
